<a href="https://colab.research.google.com/github/maddi-venkata-teja/Algoverse-Chatbot/blob/main/A24126552142_tanusree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rouge-score nltk

  Preparing metadata (setup.py) ... done


In [ ]:
import re
import math
import nltk
from collections import Counter
from rouge_score import rouge_scorer
from typing import List, Dict, Tuple

nltk.download("punkt",      quiet=True)
nltk.download("punkt_tab",  quiet=True)

print("=" * 55)
print("  ROUGE / BLEU EVALUATION  —  [tanusree]  [A24126552142]")
print("  Week 4: Text Evaluation")
print("=" * 55)
print("Imports and downloads done!")

  ROUGE / BLEU EVALUATION  —  [tanusree]  [A24126552142]
  Week 4: Text Evaluation
Imports and downloads done!


In [ ]:
REFERENCE_ANSWERS = {
    "binary_search": (
        "Binary search is an efficient algorithm for finding an item "
        "in a sorted list. It works by repeatedly dividing the search "
        "interval in half. The time complexity is O(log n)."
    ),
    "quick_sort": (
        "Quick sort is a divide and conquer sorting algorithm. "
        "It picks a pivot element and partitions the array around it. "
        "The average time complexity of quick sort is O(n log n)."
    ),
    "dynamic_programming": (
        "Dynamic programming is a method for solving complex problems "
        "by breaking them into overlapping sub-problems. It stores "
        "results of sub-problems to avoid recomputation. "
        "It is used in optimization problems."
    ),
    "linked_list": (
        "A linked list is a linear data structure where each element "
        "is a node containing data and a pointer to the next node. "
        "Insertion and deletion are O(1) at the head."
    ),
    "stack": (
        "A stack is a linear data structure that follows the "
        "Last In First Out principle. Elements are pushed and popped "
        "from the top. It is used in recursion and expression evaluation."
    ),
}

CANDIDATE_ANSWERS = {
    "binary_search": (
        "Binary search finds an element in a sorted array by dividing "
        "the interval in half repeatedly. It has O(log n) time complexity "
        "and works only on sorted data."
    ),
    "quick_sort": (
        "Quick sort is a fast sorting algorithm that uses divide and conquer. "
        "A pivot is chosen and elements are rearranged around it. "
        "It runs in O(n log n) on average."
    ),
    "dynamic_programming": (
        "Dynamic programming breaks problems into sub-problems and "
        "stores their solutions to prevent repeated computation. "
        "It is useful in optimization and scheduling."
    ),
    "linked_list": (
        "Linked list is a data structure made of nodes where each node "
        "holds data and a reference to the next node. "
        "It allows efficient insertion at the beginning."
    ),
    "stack": (
        "Stack follows LIFO order where the last inserted element "
        "is removed first. Push and pop operations work at the top. "
        "Stacks are used in function calls and parsing."
    ),
}

BLEU_WEIGHTS = {
    "bleu_1": (1.0, 0, 0, 0),
    "bleu_2": (0.5, 0.5, 0, 0),
    "bleu_4": (0.25, 0.25, 0.25, 0.25),
}

print(f"Reference answers loaded : {len(REFERENCE_ANSWERS)}")
print(f"Candidate answers loaded : {len(CANDIDATE_ANSWERS)}")
print(f"DSA topics               : {list(REFERENCE_ANSWERS.keys())}")

Reference answers loaded : 5
Candidate answers loaded : 5
DSA topics               : ['binary_search', 'quick_sort', 'dynamic_programming', 'linked_list', 'stack']


In [ ]:
# ==================== 1. ROUGE-1 ====================
def compute_rouge1(reference: str, candidate: str) -> Dict:
    scorer  = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
    scores  = scorer.score(reference, candidate)
    result  = {
        "precision" : round(scores["rouge1"].precision, 4),
        "recall"    : round(scores["rouge1"].recall,    4),
        "f1"        : round(scores["rouge1"].fmeasure,  4),
    }
    print(f"[rouge1] Precision: {result['precision']} | Recall: {result['recall']} | F1: {result['f1']}")
    return result

# ==================== 2. ROUGE-2 ====================
def compute_rouge2(reference: str, candidate: str) -> Dict:
    scorer  = rouge_scorer.RougeScorer(["rouge2"], use_stemmer=True)
    scores  = scorer.score(reference, candidate)
    result  = {
        "precision" : round(scores["rouge2"].precision, 4),
        "recall"    : round(scores["rouge2"].recall,    4),
        "f1"        : round(scores["rouge2"].fmeasure,  4),
    }
    print(f"[rouge2] Precision: {result['precision']} | Recall: {result['recall']} | F1: {result['f1']}")
    return result

# ==================== 3. ROUGE-L ====================
def compute_rougel(reference: str, candidate: str) -> Dict:
    scorer  = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores  = scorer.score(reference, candidate)
    result  = {
        "precision" : round(scores["rougeL"].precision, 4),
        "recall"    : round(scores["rougeL"].recall,    4),
        "f1"        : round(scores["rougeL"].fmeasure,  4),
    }
    print(f"[rougeL] Precision: {result['precision']} | Recall: {result['recall']} | F1: {result['f1']}")
    return result

# ==================== 4. BLEU ====================
def compute_bleu(reference: str, candidate: str, n: int = 4) -> Dict:
    ref_tokens  = nltk.word_tokenize(reference.lower())
    cand_tokens = nltk.word_tokenize(candidate.lower())
    scores = {}
    for name, weights in BLEU_WEIGHTS.items():
        score = nltk.translate.bleu_score.sentence_bleu(
            [ref_tokens],
            cand_tokens,
            weights            = weights,
            smoothing_function = nltk.translate.bleu_score.SmoothingFunction().method1,
        )
        scores[name] = round(score, 4)
    c  = len(cand_tokens)
    r  = len(ref_tokens)
    bp = 1.0 if c >= r else math.exp(1 - r / c)
    scores["brevity_penalty"] = round(bp, 4)
    print(f"[bleu] BLEU-1: {scores['bleu_1']} | BLEU-2: {scores['bleu_2']} | BLEU-4: {scores['bleu_4']} | BP: {scores['brevity_penalty']}")
    return scores

# ==================== 5. N-GRAM PRECISION ====================
def compute_ngram_precision(reference: str, candidate: str, n: int) -> float:
    ref_tokens  = nltk.word_tokenize(reference.lower())
    cand_tokens = nltk.word_tokenize(candidate.lower())
    def get_ngrams(tokens, n):
        return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    ref_ngrams  = Counter(get_ngrams(ref_tokens,  n))
    cand_ngrams = Counter(get_ngrams(cand_tokens, n))
    matches     = sum((cand_ngrams & ref_ngrams).values())
    total       = sum(cand_ngrams.values())
    precision   = matches / total if total > 0 else 0.0
    print(f"[ngram_precision] n={n} | matches: {matches} | total: {total} | precision: {round(precision, 4)}")
    return round(precision, 4)

# ==================== 6. EVALUATE ALL TOPICS ====================
def evaluate_all_topics(references: Dict[str, str], candidates: Dict[str, str]) -> Dict[str, Dict]:
    all_results = {}
    for topic in references:
        print(f"\n  Topic: {topic.upper().replace('_', ' ')}")
        print("  " + "-" * 45)
        ref  = references[topic]
        cand = candidates[topic]
        all_results[topic] = {
            "rouge1" : compute_rouge1(ref, cand),
            "rouge2" : compute_rouge2(ref, cand),
            "rougeL" : compute_rougel(ref, cand),
            "bleu"   : compute_bleu(ref,   cand),
        }
    print(f"\n[evaluate_all_topics] Evaluated {len(all_results)} topic(s).")
    return all_results

# ==================== 7. BEST AND WORST ====================
def find_best_worst(all_results: Dict[str, Dict], metric: str = "rouge1") -> Tuple[str, str]:
    scores = {}
    for topic, result in all_results.items():
        if metric in ("rouge1", "rouge2", "rougeL"):
            scores[topic] = result[metric]["f1"]
        else:
            scores[topic] = result["bleu"]["bleu_4"]
    best  = max(scores, key=scores.get)
    worst = min(scores, key=scores.get)
    print(f"[find_best_worst] Metric: {metric}")
    print(f"  Best topic  : {best}  (score: {scores[best]})")
    print(f"  Worst topic : {worst} (score: {scores[worst]})")
    return best, worst

# ==================== 8. SINGLE PAIR ====================
def evaluate_single_pair(reference: str, candidate: str, label: str = "custom") -> Dict:
    print(f"\n[evaluate_single_pair] Label: {label}")
    print(f"  Reference : {reference[:70]}...")
    print(f"  Candidate : {candidate[:70]}...")
    print("  " + "-" * 45)
    result = {
        "rouge1" : compute_rouge1(reference, candidate),
        "rouge2" : compute_rouge2(reference, candidate),
        "rougeL" : compute_rougel(reference, candidate),
        "bleu"   : compute_bleu(reference,   candidate),
    }
    return result

# ==================== 9. STATS ====================
def print_evaluation_stats(all_results: Dict[str, Dict]) -> None:
    print("\n" + "=" * 55)
    print("  ROUGE / BLEU EVALUATION STATS")
    print("=" * 55)
    rouge1_f1s = []
    rouge2_f1s = []
    rougel_f1s = []
    bleu4s     = []
    for topic, result in all_results.items():
        r1 = result["rouge1"]["f1"]
        r2 = result["rouge2"]["f1"]
        rl = result["rougeL"]["f1"]
        b4 = result["bleu"]["bleu_4"]
        rouge1_f1s.append(r1)
        rouge2_f1s.append(r2)
        rougel_f1s.append(rl)
        bleu4s.append(b4)
        label = topic.replace("_", " ").title()
        print(f"\n  {label}")
        print(f"    ROUGE-1 F1 : {r1}")
        print(f"    ROUGE-2 F1 : {r2}")
        print(f"    ROUGE-L F1 : {rl}")
        print(f"    BLEU-1     : {result['bleu']['bleu_1']}")
        print(f"    BLEU-2     : {result['bleu']['bleu_2']}")
        print(f"    BLEU-4     : {b4}")
        print(f"    Brevity P  : {result['bleu']['brevity_penalty']}")
    print("\n" + "=" * 55)
    print("  AVERAGES ACROSS ALL TOPICS")
    print("=" * 55)
    print(f"  Avg ROUGE-1 F1 : {round(sum(rouge1_f1s)/len(rouge1_f1s), 4)}")
    print(f"  Avg ROUGE-2 F1 : {round(sum(rouge2_f1s)/len(rouge2_f1s), 4)}")
    print(f"  Avg ROUGE-L F1 : {round(sum(rougel_f1s)/len(rougel_f1s), 4)}")
    print(f"  Avg BLEU-4     : {round(sum(bleu4s)/len(bleu4s),         4)}")
    print("=" * 55)

print("All functions defined successfully!")

All functions defined successfully!


In [ ]:
# Step 1 — ROUGE-1 on Binary Search
print("\n--- Step 1: ROUGE-1 Score (Binary Search) ---")
r1_result = compute_rouge1(
    REFERENCE_ANSWERS["binary_search"],
    CANDIDATE_ANSWERS["binary_search"],
)

# Step 2 — ROUGE-2 on Binary Search
print("\n--- Step 2: ROUGE-2 Score (Binary Search) ---")
r2_result = compute_rouge2(
    REFERENCE_ANSWERS["binary_search"],
    CANDIDATE_ANSWERS["binary_search"],
)

# Step 3 — ROUGE-L on Binary Search
print("\n--- Step 3: ROUGE-L Score (Binary Search) ---")
rl_result = compute_rougel(
    REFERENCE_ANSWERS["binary_search"],
    CANDIDATE_ANSWERS["binary_search"],
)

# Step 4 — BLEU Score on Binary Search
print("\n--- Step 4: BLEU Score (Binary Search) ---")
bleu_result = compute_bleu(
    REFERENCE_ANSWERS["binary_search"],
    CANDIDATE_ANSWERS["binary_search"],
)

# Step 5 — N-gram Precision Breakdown
print("\n--- Step 5: N-gram Precision Breakdown (Binary Search) ---")
for n in [1, 2, 3, 4]:
    compute_ngram_precision(
        REFERENCE_ANSWERS["binary_search"],
        CANDIDATE_ANSWERS["binary_search"],
        n=n,
    )

# Step 6 — Evaluate All Topics
print("\n--- Step 6: Evaluate All DSA Topics ---")
all_results = evaluate_all_topics(REFERENCE_ANSWERS, CANDIDATE_ANSWERS)

# Step 7 — Best and Worst Topics
print("\n--- Step 7: Best and Worst Topics ---")
best, worst = find_best_worst(all_results, metric="rouge1")

# Step 8 — Custom Single Pair
print("\n--- Step 8: Single Pair Evaluation (Custom) ---")
custom_ref  = (
    "A graph is a non-linear data structure consisting of vertices "
    "and edges. It is used to represent networks and relationships."
)
custom_cand = (
    "Graph is a data structure with nodes and edges used to "
    "represent connections between objects in a network."
)
single_result = evaluate_single_pair(custom_ref, custom_cand, label="Graph DSA")

# Step 9 — Full Stats
print("\n--- Step 9: Full Evaluation Stats ---")
print_evaluation_stats(all_results)

print("\nWeek 4 — ROUGE / BLEU Evaluation complete!")


--- Step 1: ROUGE-1 Score (Binary Search) ---
[rouge1] Precision: 0.7241 | Recall: 0.6774 | F1: 0.7

--- Step 2: ROUGE-2 Score (Binary Search) ---
[rouge2] Precision: 0.3571 | Recall: 0.3333 | F1: 0.3448

--- Step 3: ROUGE-L Score (Binary Search) ---
[rougeL] Precision: 0.5517 | Recall: 0.5161 | F1: 0.5333

--- Step 4: BLEU Score (Binary Search) ---
[bleu] BLEU-1: 0.6641 | BLEU-2: 0.4769 | BLEU-4: 0.2125 | BP: 0.9131

--- Step 5: N-gram Precision Breakdown (Binary Search) ---
[ngram_precision] n=1 | matches: 24 | total: 33 | precision: 0.7273
[ngram_precision] n=2 | matches: 12 | total: 32 | precision: 0.375
[ngram_precision] n=3 | matches: 5 | total: 31 | precision: 0.1613
[ngram_precision] n=4 | matches: 2 | total: 30 | precision: 0.0667

--- Step 6: Evaluate All DSA Topics ---

  Topic: BINARY SEARCH
  ---------------------------------------------
[rouge1] Precision: 0.7241 | Recall: 0.6774 | F1: 0.7
[rouge2] Precision: 0.3571 | Recall: 0.3333 | F1: 0.3448
[rougeL] Precision: 0.551